### Imports

In [ ]:
import os
import pandas as pd
import json
from tokenizers import Tokenizer # main object for tokenization
from tokenizers.models import BPE # model implementing Byte Pair Encoding.
from tokenizers.trainers import BpeTrainer # trains the BPE vocabulary
from tokenizers.pre_tokenizers import Whitespace # simple pre-tokenizer splitting on spaces

### Settings

In [2]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

### Tokenization

In [3]:
tokenizer = Tokenizer(BPE())           # BPE model with unknown token
tokenizer.pre_tokenizer = Whitespace() # Split text by spaces first

In [4]:
# Load the cleaned corpus
with open(os.path.join(DIR_DATA, "cleaned_corpus.txt"), "r", encoding="utf-8") as f:
    corpus = f.read()

In [19]:
# Training the tokenizer on the corpus.
VOCAB_SIZE = 25000          # Educated guess based on the number of unique tokens and the long tail distribution of token counts
MIN_FREQUENCY = 1          # Minimum frequency for a token to be included in the vocabulary, based on the distribution of token counts and the large number of singular tokens in the corpus
SPECIAL_TOKENS = []         # No special tokens for now, but we could add [UNK], [PAD], etc. if needed
END_OF_WORD_SUFFIX = "</w>" # Suffix to indicate the end of a word, commonly used in BPE tokenization to distinguish between subword tokens and whole word tokens
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, min_frequency=MIN_FREQUENCY, special_tokens=SPECIAL_TOKENS, end_of_word_suffix=END_OF_WORD_SUFFIX)

# FIX: Wrap corpus in a list so it is treated as a single document/sequence, not a list of characters
tokenizer.train_from_iterator([corpus], trainer=trainer)

vocab_df = pd.DataFrame(tokenizer.get_vocab().items(), columns=["token", "id"]).sort_values("id").reset_index(drop=True)
print(f"Sample of the learned BPE vocabulary:")
print(vocab_df.sample(10))

Sample of the learned BPE vocabulary:
                  token     id
7309         arming</w>   7309
15020         dante</w>  15020
16992  continuously</w>  16992
17995       reveals</w>  17995
14176           willing  14176
12141    accidental</w>  12141
6603        torture</w>   6603
12205       agonies</w>  12205
4262                pit   4262
5421         thetic</w>   5421


In [20]:
# Example of encoding some words using the trained tokenizer
print(f"Encoding some sample words using the trained BPE tokenizer:")
for words in ["bathroom", "scientology", "lowest", "absolutely", "nocap", "threehundredandthirtythreetrillionth"]:
    print(f"    {words:<36} => {'-'.join(tokenizer.encode(words).tokens)}")

Encoding some sample words using the trained BPE tokenizer:
    bathroom                             => ba-th-room</w>
    scientology                          => sci-ent-o-logy</w>
    lowest                               => lowest</w>
    absolutely                           => absolutely</w>
    nocap                                => no-cap</w>
    threehundredandthirtythreetrillionth => thre-e-hundre-dan-d-thir-ty-thre-e-tri-lli-on-th</w>


In [21]:
# Save then trained tokenizer
tokenizer.save("bpe_tokenizer.json")

In [22]:
# Encode the corpus using the trained tokenizer (could take a couple minutes)
# " ".join(corpus) was inserting spaces between every character if corpus is a string!
# If corpus is already a string, just pass it directly.
encoded_corpus = tokenizer.encode(corpus)

In [30]:
# Try detokenizing a sample of the encoded corpus to verify it works correctly
sample_encoded = encoded_corpus.ids[:100] # Take the first 100 token IDs as a sample
detokenized_sample = tokenizer.decode(sample_encoded)  
print(f"\nSample of the encoded corpus (first 100 token IDs):\n{sample_encoded}")
print(f"\nDetokenized sample:\n{detokenized_sample}")


Sample of the encoded corpus (first 100 token IDs):
[79, 2957, 162, 1603, 3465, 1946, 5249, 1289, 1289, 1289, 1289, 1289, 142, 239, 7210, 86, 871, 109, 232, 12642, 341, 3173, 79, 11375, 84, 121, 9997, 172, 142, 167, 3019, 126, 331, 2084, 65, 2021, 519, 82, 143, 417, 4093, 110, 86, 2869, 143, 782, 1785, 84, 143, 9650, 82, 7073, 3452, 94, 79, 3300, 84, 143, 65, 324, 947, 730, 1202, 84, 82, 91, 65, 1964, 94, 79, 4179, 84, 65, 1064, 54, 1707, 6793, 8570, 1902, 345, 143, 172, 8175, 128, 143, 7481, 82, 9506, 116, 126, 280, 142, 1163, 174, 174, 172, 341, 5791, 185, 79]

Detokenized sample:
the</w> modern</w> by</w> mary</w> shel ley</w> contents</w> letter</w> letter</w> letter</w> letter</w> letter</w> you</w> will</w> rejoice</w> to</w> hear</w> that</w> no</w> disaster</w> has</w> accompanied</w> the</w> commencement</w> of</w> an</w> enterprise</w> which</w> you</w> have</w> regarded</w> with</w> such</w> evil</w> p</w> arrived</w> here</w> and</w> my</w> first</w> task</w> is</w> to</w>

In [29]:
# Save the encoded corpus
with open("encoded_text.json", "w") as f:
    json.dump(encoded_corpus.ids, f)